In [3]:
from langgraph.graph import StateGraph
from typing import TypedDict
import os 
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from chat_memory import get_chat_history
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import HumanMessage
from video_processing import analyze_video
from vectorstore_setup import clean_classification_text
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langgraph.graph import StateGraph, START, END
import tempfile

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = Chroma(persist_directory="path/to/your/chroma_db", embedding_function=embeddings)

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")



# when a query comes in, use this model to embed the query text so you can compare it against the vectors already stored.
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = Chroma(persist_directory="/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/chroma_db", 
                                    embedding_function=embeddings)


# Whisper speach to text 
from openai import OpenAI
client = OpenAI()




In [4]:
router_prompt = ChatPromptTemplate.from_messages([
    ("system", """Your job is to classify user queries into two buckets: 
     - memory
     - memory and vectorstore
     
     *Guidelines*
     Questions regarding exercises, exercise form, or technique: vectorstore & memory
     General question or statements: memory
   
     *Respond to queries with only the correct class*
     Examples: 
     User: how should I grip the bar for bench press?
     Agent: vectorstore & memory

     User: what muscles should I feel during the Romanian dead lift?
     Agent: vectorstore & memory

     User: How's this? 
     Agent: vectorstore & memory

     User: Is this good squat technique? 
     Agent: vectorstore & memory

     User: Was my bar path better?
     Agent: vectorstore & memory

     User: thakns for your help!
     Agent: memory

"""),

     ("human", "{query}")
     
])

llm = ChatOpenAI(model='gpt-4o')

output_parser = StrOutputParser()

router_chain = router_prompt | llm | output_parser


In [5]:
# Define the state

# the state is a dictionary

class GraphState(TypedDict):

    session_id: int 

    user_query: str

    user_video: str

    classification_image: str

    classified_keywords: str

    top_k_chunks: str

    encoded_images: list[str]

    response: str




In [6]:
workflow = StateGraph(GraphState) 

In [7]:
def video_encoder_node(state: GraphState):
    user_video = state["user_video"]
    user_video_encoded = analyze_video(filepath_in=user_video, frame_count=15, max_seconds=10) # encodes video
    encoded_images = [] # creates a list of encoded images for LLM
    for images in user_video_encoded:
        encoded_images.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{images}"}})
        classification_image = user_video_encoded[0] 
    
    return {"classification_image": classification_image, "encoded_images": encoded_images}


In [8]:
# video classification router NODE

def video_classification_node(state: GraphState):
    classification_image = state["classification_image"]

    router_llm = ChatOpenAI(model='gpt-4o')

    response = router_llm.invoke([

    {"role": "user", "content": 

    [{"type": "text", "text":  """Your job is to analyze images of users working out for proper form, and list the key checkpoints of their to body evaluate. 
    Give me ONLY the bodypart checkpoints. Do NOT include evaluation suggestions. Do NOT include an intro sentence. 
    Output format should be exactly the example below.
    **Example**
    Overhead press

    1. Feet & base
    2. Glutes & legs
    3. Core & Ribcage
    4. Shoulder position
    5. Bar path
    6. Head & Neck
    7. Lockout position
    8. Tempo and control
    """},


        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{classification_image}"}}
        ]}


    ])
    print(response.text)

    classified_keywords = clean_classification_text(response)

    return {"classified_keywords": classified_keywords}


In [9]:
def vector_db_node(state: GraphState):
    user_query = state["user_query"]
    classified_keywords = state.get("classified_keywords", "")
    retrieval_query = classified_keywords

    if classified_keywords:
        results_user = vectorstore.similarity_search(user_query, k=5)
        results_video = vectorstore.similarity_search(retrieval_query, k=5)
        results = results_user + results_video

    if not classified_keywords: 
        results_user = vectorstore.similarity_search(user_query, k=5)
        results = results_user

    unique_results = []
    seen = set()

    for r in results:
        content = r.page_content
        if content not in seen:
            seen.add(content)
            unique_results.append(r)


    top_k_chunks = "\n".join([r.page_content for r in unique_results])

    return {"top_k_chunks": top_k_chunks}

In [10]:
def response_generator(state: GraphState):
    top_k_chunks = state['top_k_chunks']
    encoded_images = state.get("encoded_images", [])
    session_id = state["session_id"]
    user_query = state["user_query"]

    prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a world-class fitness coach. You have extensive experience in helping weight lifters achieve perfect form and maximum hypertrophy. 
    Your job is to analyze images of users lifting weights, offer them advice from your context, and to answer any questions they might have. 
    Inspect each image CLOSELY and arefuly for problems or issues related to best practices in exercise form. Help the user diagnose their incorrect form. 
    Be specific about what you observe.

    # ANSWER CONTEXT
    Use ONLY the following context when answering a user: 
        
    ---   
    {top_k_chunks}
    ---
    if the query or image isn't in context, reply, 'I don't have expert coaching content for this exercise yet. 
    Currently I can analyze: bench press, overhead press, incline bench press...'"
    """),
        MessagesPlaceholder(variable_name="history"),
        MessagesPlaceholder(variable_name="input"),
    ])



    user_msg = HumanMessage(content=[
        {"type": "text", "text": user_query},
        *encoded_images,   # <- your list of {"type":"image_url",...}
    ])

    llm = ChatOpenAI(model='gpt-5',
                    temperature=0.5)

    output_parser = StrOutputParser()

    chain = prompt | llm | output_parser

    chain_with_history = RunnableWithMessageHistory(
        chain,
        get_session_history=get_chat_history,
        input_messages_key="input",
        history_messages_key="history"

    )

    response = chain_with_history.invoke(
        {"input": [user_msg], "top_k_chunks": top_k_chunks},
        config={"configurable": {"session_id": session_id}}
    )


    return {"response": response}
    

In [11]:
def chat_memory(state: GraphState):
    user_query = state["user_query"]
    session_id = state["session_id"]
    
    prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a world-class fitness coach. You have extensive experience in helping weight lifters achieve perfect form and maximum hypertrophy. 
    Your job is to analyze images of users lifting weights, offer them advice from your context, and to answer any questions they might have. 


    """),
        MessagesPlaceholder(variable_name="history"),
        MessagesPlaceholder(variable_name="input"),
    ])

    user_msg = HumanMessage(content=[
        {"type": "text", "text": user_query}
    ])

    llm = ChatOpenAI(model='gpt-4o',
                    temperature=0.5)

    output_parser = StrOutputParser()

    chain = prompt | llm | output_parser

    chain_with_history = RunnableWithMessageHistory(
        chain,
        get_session_history=get_chat_history,
        input_messages_key="input",
        history_messages_key="history"

    )

    response = chain_with_history.invoke(
        {"input": [user_msg]},
        config={"configurable": {"session_id": session_id}}
    )

    return {"response": response}
    


# Router function

In [12]:
def route_query(state: GraphState):
    # first check: is there a video?
    if state["user_video"]:
        return "video_encoder"
    
    # no video — ask the LLM which path
    route = router_chain.invoke({"query": state["user_query"]})
    route = route.lower().strip()
    
    if "vectorstore" in route:
        return "vector_db"
    else:
        return "chat_memory"

# Audio transcription function

In [13]:
def transcribe_audio(audio_path):
    audio_file = open(audio_path, "rb") # rb = read binary. audio files are binary data (raw bytes). this tells python gto open as binary instaed of text
    transcription = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file
    )

    return transcription.text
    

user_query = transcribe_audio("/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/data/raw_voice_notes/bench_press_voice_note.m4a")
print(user_query)

Why does it help to keep my butt on the bench once the bar is out?


In [14]:
# conditional edge
workflow.add_conditional_edges(START, route_query)

# add nodes
workflow.add_node("video_encoder", video_encoder_node)
workflow.add_node("video_classification_router", video_classification_node)
workflow.add_node("vector_db", vector_db_node)
workflow.add_node("response_generator", response_generator)
workflow.add_node("chat_memory", chat_memory)



# connect them
workflow.add_edge("chat_memory", END)
workflow.add_edge("video_encoder", "video_classification_router")
workflow.add_edge("video_classification_router", "vector_db")
workflow.add_edge("vector_db", "response_generator")
workflow.add_edge("response_generator", END)

app = workflow.compile()


In [15]:
result = app.invoke({
    "session_id": 123,
    "user_query": "how's my form?",
    "user_video": "/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/data/raw_workout_videos/bench_press/bench_07.mov"
})

print(result["response"])

Frames processed: 15, (10s cap)
Saved to processed-images/bench_07
Video name: bench_07
Bench Press

1. Feet & base
2. Glutes & legs
3. Lower back arch
4. Core & Ribcage
5. Shoulder position
6. Hand grip
7. Bar path
8. Head & Neck
9. Lockout position
Thanks for the video — this is a flat barbell bench press. Here’s what I’m seeing and how to tighten it up:

What looks good
- You’ve set a noticeable arch and you’re touching the bar to the chest each rep.
- Tempo looks controlled.

Biggest fixes
- Glutes: Your butt appears to lift off the pad during the press. You cannot raise your butt up off the bench at any time. Keep four points of contact planted: head, upper back, glutes, and feet.
- Foot/leg drive: Your heels are up and your feet are set forward. For regulation technique, plant your heels and pull your feet back as far as you comfortably can so the shins are nearly vertical. Keep the legs close to the bench when viewed from the front. Drive through the heels as you press.
- Upper-

In [ ]:

# result = app.invoke({
#     "session_id": 123,
#     "user_query": user_query,
#     "user_video": ""
# })

# print(result["response"])

Great question. Keeping your butt planted matters because:

- It lets your leg drive actually travel along the bench through the hips into the arched back to reinforce the arch and keep the chest in its high position. That’s how you stay tighter and press more efficiently.
- Heels-down leg drive works best when your hips stay down and your shins are nearly vertical. Lifting the hips turns it into a “bridge,” which breaks that force transfer.
- If your butt lifts or slides, your core goes soft, you lose your footing, and things can get dangerous—especially on heavy or grinder reps. If the bench is slippery, add some bungees so your butt sticks.

So: heels planted, hips down, squeeze the glutes, and drive—without letting the butt leave the pad.


In [17]:
result = app.invoke({
    "session_id": 123,
    "user_query": "thanks for your help today!",
    "user_video": ""
})

print(result["response"])

You're welcome! If you have any more questions or need further assistance, feel free to ask. Happy lifting! 🏋️‍♂️


In [ ]:
# result = app.invoke({
#     "session_id": 123,
#     "user_query": "can you tell me again how to make it easier?",
#     "user_video": ""
# })

# print(result["response"])

Absolutely. Here’s how to make your bench feel easier right away:

- Keep your feet planted—no “floppy feet.” Tight legs help you stay tight everywhere and press more effectively. If your core or lower body loosens up, reset before the next rep; don’t press with loose legs and a loose core.
- Fix a slippery bench. If your butt slides, your core goes soft and you lose footing. Wrap a few bungees around the seat so your butt sticks and you can stay tight, especially on heavy reps.
- Unrack efficiently. Take a deep breath, brace, drive your shoulder blades/back hard into the bench so the bar “comes off” the uprights rather than you lifting it up. Saves energy for the set.
- Use the right bar path. Press the bar up and back toward the rack, not straight up. Accelerate off the chest. If you hit a sticking point, you can flare the elbows a bit more to get the pecs to help the triceps.
- Build gradually. Start with lighter weight, focus on clean reps, and build it up over time. The simplest l

In [19]:
def correctness_evaluator(run, example):
    generated = run.outputs["answer"]
    reference = example.outputs["answer"]
    question = example.inputs["user_query"]

    prompt = f"""You are evaluating a fitness coaching AI assistant.
Your job is to assess the quality of its response against a reference answer.

Question: {question}
Reference answer: {reference}
Generated answer: {generated}

Scoring criteria:

Score 0 — The generated answer contains incorrect or potentially harmful advice that contradicts the reference answer. This is the worst outcome.
Score 1 — The generated answer is incomplete or omits important details from the reference, but contains nothing incorrect or harmful.
Score 2 — The generated answer is correct and captures the key facts from the reference answer.

Important: Penalize incorrect advice more harshly than omission. A response that says nothing is better than one that says something wrong.

Reply with only the number: 0, 1, or 2."""

    from openai import OpenAI
    openai_client = OpenAI()
    result = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}]
    )
    try: 
        score_raw = result.choices[0].message.content.strip()
        score = int(''.join(filter(str.isdigit, score_raw))[0])
    except:
        score = -1
    return {"key": "correctness", "score": score / 2} # normalize the score so it's <= 1

In [20]:
def run_rag_pipeline(inputs: dict, attachments: dict) -> dict:
    # grab video from attachments in LangSmith
    video_name = list(attachments.keys())[0] 
    # read the video as raw bytes
    video_bytes = attachments[video_name]["reader"].read()
    
    # save to temp file because analyze_video expects a filepath
    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as tmp:
        # . delete=False means "don't delete this file when we're done writing" because we still need it for the pipeline.
        tmp.write(video_bytes)
        tmp_path = tmp.name
    
    state = {
        "user_query": inputs["user_query"],
        "session_id": "eval-session",
        "classified_keywords": "",
        "encoded_images": [],
        "user_video": tmp_path
    }
    
    state = {**state, **video_encoder_node(state)}
    state = {**state, **video_classification_node(state)}
    state = {**state, **vector_db_node(state)}
    state = {**state, **response_generator(state)}

    print("STATE KEYS:", state.keys())
    print("RESPONSE:", state.get("response", "NOT FOUND"))
    
    return {"answer": state["response"]}

In [21]:
from langsmith import Client
from langsmith.evaluation import evaluate

langsmith_client = Client()

os.environ["LANGCHAIN_TRACING_V2"] = "false" 
# stops LangSmith from trying to log all the big intermediate data
# evaluate() function will still capture your inputs, outputs, and evaluator scores
# it just won't try to upload the full trace with all those heavy frames.

results = evaluate(
    run_rag_pipeline,
    data="Image assessment response accuracy v1",
    evaluators=[correctness_evaluator],
    experiment_prefix="vision-faithfullness-rag-v1"
)

View the evaluation results for experiment: 'vision-faithfullness-rag-v1-93166778' at:
https://smith.langchain.com/o/5dffb1fc-82e5-4000-a9a5-d9c923e0f73c/datasets/1be4093e-6f7d-4021-a94b-d6cf76c22b6a/compare?selectedSessions=46b314d4-fc00-4665-9b2e-7b23a3271bda




0it [00:00, ?it/s]

Frames processed: 15, (10s cap)
Saved to processed-images/tmp7uzjftxc
Video name: tmp7uzjftxc
**Bench Press**

1. Feet & base
2. Glutes & legs
3. Core & lower back
4. Shoulder position
5. Grip & wrist alignment
6. Elbow position
7. Bar path
8. Head & neck
STATE KEYS: dict_keys(['user_query', 'session_id', 'classified_keywords', 'encoded_images', 'user_video', 'classification_image', 'top_k_chunks', 'response'])
RESPONSE: Thanks for the clips—here’s what I’m seeing and how to dial it in.

What looks good
- You’re under control and keeping your butt on the bench.
- Bar path is generally consistent rep to rep.

Key fixes (in order of impact)
- Foot placement/leg drive: Your feet look a bit loose and set wide from the bench. Bring them back as far as you can while still keeping them flat if you go heels-down, or commit to a solid toes style—but either way, keep the legs tight and close to the bench when viewed from the front. Push the knees out, squeeze the glutes, and keep that lower body

Error running target function: [Errno 2] No such file or directory: '/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/data/processed/processed-images/tmpjdiluac7_14.jpg'
Traceback (most recent call last):
  File "/opt/miniconda3/envs/fitness-form-coach/lib/python3.10/site-packages/langsmith/evaluation/_runner.py", line 1903, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "/opt/miniconda3/envs/fitness-form-coach/lib/python3.10/site-packages/langsmith/run_helpers.py", line 758, in wrapper
    function_result = run_container["context"].run(
  File "/var/folders/2n/c95ntf5d1wx1_vrd1b3ljbkw0000gn/T/ipykernel_48184/719908470.py", line 21, in run_rag_pipeline
    state = {**state, **video_encoder_node(state)}
  File "/var/folders/2n/c95ntf5d1wx1_vrd1b3ljbkw0000gn/T/ipykernel_48184/436431845.py", line 3, in video_encoder_node
    user_video_encoded = analyze_video(filepath_in=user_video, frame_count=15, max_seconds=10) # encodes video
  File "/Users/chandlershor

Frames processed: 15, (10s cap)
Saved to processed-images/tmpjdiluac7
Video name: tmpjdiluac7
Frames processed: 15, (10s cap)
Saved to processed-images/tmpchylxsux
Video name: tmpchylxsux
Bench Press

1. Feet & base
2. Glutes & legs
3. Lower back arch
4. Shoulder position
5. Elbow alignment
6. Bar path
7. Wrist position
8. Tempo and control
STATE KEYS: dict_keys(['user_query', 'session_id', 'classified_keywords', 'encoded_images', 'user_video', 'classification_image', 'top_k_chunks', 'response'])
RESPONSE: Nice work getting controlled reps. A few tweaks will make your bench stronger and more consistent.

What I see
- Feet look wide and a bit loose; you’re not getting much leg drive.
- Upper back looks “flat” on the pad after the unrack.
- Elbows flare and the touch point looks a bit high on your chest.
- Wrists appear bent back at lockout.

Fix-it cues for your next set
- Build a base: keep four points of contact—head, upper back, glutes, and feet. Set your feet back as far as you can 

Failed to send compressed multipart ingest: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. The content length of 189594468 bytes exceeds the maximum size limit of 20971520 bytes. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/multipart (Caused by SSLError(SSLError(5, '[SYS] unknown error (_ssl.c:2426)')))"))
Content-Length: 189594468
API Key: lsv2_********************************************cctrace=019cdc8e-cf53-7f52-a053-f0a7ae460c40,id=019cdc8e-dfcd-7250-9b94-50723c32984d; trace=019cdc8e-cf53-7f52-a053-f0a7ae460c40,id=019cdc8e-dfd0-7b91-815f-6fbb4b321e3f; trace=019cdc8e-cf53-7f52-a053-f0a7ae460c40,id=019cdc8e-dfd2-7d83-9417-baf399cc9e59; trace=019cdc8e-cf53-7f52-a053-f0a7ae460c40,id=019cdc8e-dfda-7702-8298-129841db7e2d; trace=019cdc8e-cf53-7f52-a053-f0a7ae460c40,id=019cdc8e-dfda-7702-8298-129841db7e2d; trace=019cdc8e-cf53-7f52-a053-f0a7ae460c40,id=019

STATE KEYS: dict_keys(['user_query', 'session_id', 'classified_keywords', 'encoded_images', 'user_video', 'classification_image', 'top_k_chunks', 'response'])
RESPONSE: Good set—controlled reps and your hips stay on the pad. A few fixes will make it stronger and safer.

What I see
- Feet are forward and a bit loose, so you’re not getting much leg drive.
- Upper back looks a little “flat” after the liftoff.
- Elbows flare and the touch point looks slightly high on your chest.
- Wrists are bent back behind the bar even with wraps.

Do this on your next set
- Set your base: eyes under the bar, pinch your shoulder blades together and keep them retracted the whole set. Keep four points of contact—head, upper back, glutes, and feet.
- Foot position/leg drive: bring your feet back as far as you comfortably can while keeping them planted and close to the bench when viewed from the front. Push the knees out slightly, squeeze the glutes, and keep the legs tight throughout the set.
- Elbow path a

Failed to send compressed multipart ingest: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. Please confirm your internet connection. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/multipart (Caused by SSLError(SSLError(5, '[SYS] unknown error (_ssl.c:2426)')))"))
Content-Length: 19953592
API Key: lsv2_********************************************cctrace=019cdc8e-cf53-7f52-a053-f0a7ae460c40,id=019cdc8e-e003-7b13-93a6-19d64b2a8eea
Failed to send compressed multipart ingest: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. Please confirm your internet connection. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/multipart (Caused by SSLError(SSLError(5, '[SYS] unknown error (_ssl.c:2426)')))"))
Content-Length: 19945487
API Key: ls